# Валідація

**Кроки:** завантаження → поділ → склеювання → порівняння з тестом → метрика → baseline → збереження.

In [ ]:
import os, re, ast, string
from typing import List
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import GroupShuffleSplit, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED      = 21
N_BIG     = 200_000
N_DOCS    = 2000
UK_SHARE  = 0.9
MIN_LEN   = 100

files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]

def find(word, exts=('.csv',)):
    return [f for f in files if word in os.path.basename(f).lower() and f.endswith(exts)][0]

def load(path, n=None):
    return pd.read_csv(path, nrows=n)

def to_list(x):
    if isinstance(x, str):
        x = ast.literal_eval(x)
    return [tuple(s) for s in x]

uk   = load(find('uk_geo_dataset.csv'), N_BIG)
ru   = load(find('ru_geo'), N_BIG)
lab  = load(find('labeling'))
test = load(find('test'))

for df in (uk, ru, lab):
    df['loc_markers'] = df['loc_markers'].apply(to_list)
    df['locs'] = df.apply(lambda r: [r['text'][a:b] for a, b in r['loc_markers']], axis=1)

uk['source'], ru['source'] = 'uk', 'ru'
uk['row'], ru['row'] = uk.index, ru.index
print(len(uk), len(ru), len(lab), len(test))

200000 200000 26 477


## 1. Поділ на train / valid по групах

In [ ]:
ru['group'] = 'ru_' + ru['doc_id'].astype(str)
ru['order'] = ru['sent_id']
uk['group'] = 'uk_' + (uk['row'] // 100).astype(str)
uk['order'] = uk['row']

big = pd.concat([uk, ru], ignore_index=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=SEED)
_, valid_idx = next(gss.split(big, groups=big['group']))
big['split'] = 'train'
big.loc[valid_idx, 'split'] = 'valid'

valid_texts = set(big.loc[big['split'] == 'valid', 'text'])
dup = (big['split'] == 'train') & big['text'].isin(valid_texts)
big.loc[dup, 'split'] = 'drop'

print(big.groupby(['source', 'split']).size().unstack(fill_value=0))
print('прибрано дублікатів з train:', int(dup.sum()))

split   drop   train  valid
source                     
ru      4207  186106   9687
uk        16  187684  12300
прибрано дублікатів з train: 4223


## 2. Склеювання документів для valid

In [ ]:
rng = np.random.RandomState(SEED)
target_lens = test['text'].str.len().clip(upper=1000).values

def pack_group(rows):
    docs, parts, spans, pos = [], [], [], 0
    target = rng.choice(target_lens)
    for text, markers in rows:
        new_pos = pos + len(text) + 1
        if parts and new_pos > target and (new_pos - target) > (target - pos):
            docs.append((' '.join(parts), spans))
            parts, spans, pos = [], [], 0
            target = rng.choice(target_lens)
        spans += [(a + pos, b + pos) for a, b in markers]
        parts.append(text)
        pos += len(text) + 1
        if pos >= target:
            docs.append((' '.join(parts), spans))
            parts, spans, pos = [], [], 0
            target = rng.choice(target_lens)
    if parts:
        docs.append((' '.join(parts), spans))
    return docs

valid_sent = big[big['split'] == 'valid'].sort_values(['group', 'order'])
docs = []
for group, g in valid_sent.groupby('group', sort=False):
    for text, spans in pack_group(list(zip(g['text'], g['loc_markers']))):
        docs.append({'source': group[:2], 'text': text, 'loc_markers': spans})
docs = pd.DataFrame(docs)
docs = docs[docs['text'].str.len() >= MIN_LEN]
docs['locs'] = docs.apply(lambda r: [r['text'][a:b] for a, b in r['loc_markers']], axis=1)

n_uk = int(N_DOCS * UK_SHARE)
parts = [docs[docs['source'] == 'uk'].sample(min(n_uk, (docs['source'] == 'uk').sum()), random_state=SEED),
         docs[docs['source'] == 'ru'].sample(min(N_DOCS - n_uk, (docs['source'] == 'ru').sum()), random_state=SEED)]
valid_docs = pd.concat(parts, ignore_index=True)
print('документів у valid:', valid_docs['source'].value_counts().to_dict())
print(valid_docs['text'].iloc[0][:300])
print(valid_docs['locs'].iloc[0])

документів у valid: {'uk': 1800, 'ru': 200}
Кожному потерпілому місцевому жителю також виділять ще по 20 тис. грн. Заступнику директора «Юбіай фарм трейд» вдалося зменшити суму хабара до 200 тисяч. Нині команда проекту це – архітектори, дизайнери, митці. Пізніше її знайшов один з вірян, коли копав рів для фундаменту храму. Підопічні Георгія Ц
[]


## 3. Чи схожий valid на test?

In [4]:
def lang(t):
    t = t.lower()
    u, r = bool(re.search('[іїєґ]', t)), bool(re.search('[ыэъё]', t))
    return 'укр' if u and not r else 'рос' if r and not u else 'інше'

def style(texts, locs=None):
    t = pd.Series(list(texts))
    out = {
        'символів (середнє)': t.str.len().mean(),
        'символів (медіана)': t.str.len().median(),
        '% з цифрами':   t.str.contains(r'\d').mean() * 100,
        '% з латиницею': t.str.contains('[A-Za-z]').mean() * 100,
        '% укр':         (t.apply(lang) == 'укр').mean() * 100,
    }
    if locs is not None:
        n = pd.Series([len(l) for l in locs])
        out['% без локацій'] = (n == 0).mean() * 100
        out['локацій на текст'] = n.mean()
    return pd.Series(out)

sent = big[big['split'] == 'valid']
table = pd.DataFrame({
    'valid (речення)': style(sent['text'], sent['locs']),
    'valid (склеєний)': style(valid_docs['text'], valid_docs['locs']),
    'labeling':        style(lab['text'], lab['locs']),
    'test':            style(test['text']),
})
table.round(1)

,valid (речення),valid (склеєний),labeling,test
% без локацій,74.7,42.7,38.5,NaN
% з латиницею,10.0,21.8,23.1,31.7
% з цифрами,25.3,55.0,57.7,58.5
% укр,55.2,89.6,92.3,89.5
локацій на текст,0.4,1.3,1.2,NaN
символів (медіана),95.0,300.0,217.5,264.0
символів (середнє),105.5,397.9,294.4,395.4


### Через модель (adversarial)


In [5]:
def style_features(texts):
    t = pd.Series(list(texts))
    return pd.DataFrame({
        'chars':  t.str.len(),
        'words':  t.str.split().str.len(),
        'digits': t.str.count(r'\d'),
        'latin':  t.str.count('[A-Za-z]'),
        'upper':  t.str.count(r'[A-ZА-ЯІЇЄҐ]'),
        'punct':  t.str.count(r'[^\w\s]'),
    })

def style_auc(texts_a, texts_b):
    n = min(len(texts_a), len(texts_b))
    a = pd.Series(list(texts_a)).sample(n, random_state=SEED)
    b = pd.Series(list(texts_b)).sample(n, random_state=SEED)
    X = pd.concat([style_features(a), style_features(b)], ignore_index=True)
    y = [0] * n + [1] * n
    clf = RandomForestClassifier(200, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
    p = cross_val_predict(clf, X, y, cv=3, method='predict_proba')[:, 1]
    return roc_auc_score(y, p)

print('речення       vs test:', round(style_auc(sent['text'], test['text']), 3))
print('склеєний valid vs test:', round(style_auc(valid_docs['text'], test['text']), 3))

речення       vs test: 0.917
склеєний valid vs test: 0.762


## 4. Метрика змагання

In [6]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

print('ідеальні відповіді:', comp_metric(valid_docs['locs'].tolist(), valid_docs['locs'].tolist()))
print('порожні відповіді :', comp_metric(valid_docs['locs'].tolist(), [[] for _ in range(len(valid_docs))]))

ідеальні відповіді: 1.0
порожні відповіді : 0


## 5. Baseline

In [7]:
train_df = big[big['split'] == 'train']
counter = Counter(l.lower().strip() for locs in train_df['locs'] for l in locs)
dictionary = {k for k, v in counter.items() if v >= 20 and len(k) > 2}
print('слів у словнику:', len(dictionary))

def predict(text):
    words = re.findall(r'\w+', text)
    found = []
    for n in (3, 2, 1):
        for i in range(len(words) - n + 1):
            cand = ' '.join(words[i:i + n])
            if cand.lower() in dictionary:
                found.append(cand)
    return list(dict.fromkeys(found))

def evaluate(df, name):
    preds = [predict(t) for t in df['text']]
    f1 = comp_metric(df['locs'].tolist(), preds)
    boot = []
    for _ in range(100):
        idx = rng.randint(0, len(df), len(df))
        boot.append(comp_metric([df['locs'].iloc[i] for i in idx], [preds[i] for i in idx]))
    print(f'{name:18s} ({len(df):5d}): F1 = {f1:.4f}   95% інтервал {np.percentile(boot, 2.5):.3f} … {np.percentile(boot, 97.5):.3f}')

evaluate(valid_docs, 'valid (склеєний)')
evaluate(lab, 'labeling')

слів у словнику: 876
valid (склеєний)   ( 2000): F1 = 0.6905   95% інтервал 0.677 … 0.707
labeling           (   26): F1 = 0.3214   95% інтервал 0.186 … 0.413


### Що baseline пропускає на `labeling`

In [8]:
for text, gold in zip(lab['text'], lab['locs']):
    pred = predict(text)
    missed = [g for g in gold if hard_processing(g) not in {hard_processing(p) for p in pred}]
    extra  = [p for p in pred if hard_processing(p) not in {hard_processing(g) for g in gold}]
    if missed or extra:
        print('пропущено:', missed, '| зайве:', extra)

пропущено: ['Проспект Бандери'] | зайве: []
пропущено: ['Чорнобиль', 'Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Яготин', 'Баришівка'] | зайве: []
пропущено: ['проспекті Івасюка'] | зайве: []
пропущено: ['Хелма'] | зайве: []
пропущено: [] | зайве: ['Києва', 'світ', 'України', 'Україні']
пропущено: [] | зайве: ['України']
пропущено: ['ТЦ Аркадія'] | зайве: []
пропущено: ['Фастовском', 'Бучанском'] | зайве: []
пропущено: ['Троєщині'] | зайве: ['Київ']
пропущено: ['Шевченківському районі'] | зайве: ['Києва']
пропущено: ['Героїв Дніпра'] | зайве: ['Дніпра']
пропущено: [] | зайве: ['Шевченка']
пропущено: ["вул. Солом'янська, 28"] | зайве: []
пропущено: [] | зайве: ['Місяць']
пропущено: ['Васильковом'] | зайве: ['Київ']
пропущено: [] | зайве: ['Київ']
пропущено: [] | зайве: ['Київ']
пропущено: [] | зайве: ['України', 'Київ']
пропущено: ['Броварах', 'Подолі'] | зайве: []
пропущено: ['Пскові'] | зайве: []
пропущено: ['вулиці Здолбунівській', 'вул. Здолбунівська, 1'] | зайве: []


## 6. Збереження

In [9]:
big[['source', 'row', 'split']].to_csv('/kaggle/working/splits.csv', index=False)
valid_docs[['source', 'text', 'loc_markers']].to_csv('/kaggle/working/valid_docs.csv', index=False)
print(big['split'].value_counts().to_dict(), '| valid-документів:', len(valid_docs))

{'train': 373790, 'valid': 21987, 'drop': 4223} | valid-документів: 2000


## Висновки

**1. Проблема.** Тест складається з довгих текстів з кількох речень, а великі корпуси (`uk`, `ru`) це окремі короткі речення. Розмічених текстів з розподілу тесту лише 26, тож на них одних валідуватись не можна.

**2. Рішення.** Ділимо `uk` і `ru` на train / valid **по групах** (`doc_id` для `ru`, блок зі 100 рядків для `uk`), прибираємо дублікати з train (4 223 тексти, майже всі з `ru`) і з речень valid **склеюємо документи** такої ж довжини, як у тесті (2 000 документів, 90 % українських). Поділ зберігається у `splits.csv`, склеєний valid у `valid_docs.csv`, їх використовують усі наступні ноутбуки.

**3. Наскільки valid схожий на test.**

| | valid (склеєний) | labeling | test |
|---|---|---|---|
| довжина, середня (символів) | 398 | 294 | 395 |
| % з цифрами | 55,0 | 57,7 | 58,5 |
| % з латиницею | 21,8 | 23,1 | 31,7 |
| % української | 89,6 | 92,3 | 89,5 |
| % без локацій | 42,7 | 38,5 | n/a |